# AutoMark – fine-tune the handwriting model

The model being fine-tuned is **our own handwriting reader: CNN (5 layers) + BiLSTM (2 layers) + CTC**, starting from the weights AutoEx trained on the IAM dataset (`model/htr_weights.npz`).

Run the cells from top to bottom (click ▶ on each).

**Your data** — one of these:
- **`pages.zip`**: answer-sheet photos/PDFs, each with a `.txt` file of the same name holding the exact text written on it (e.g. `aditya_p1.jpg` + `aditya_p1.txt`). The part of the name before `_` is the student.
- **`training_data.zip`**: downloaded from the app (Settings → Download training_data.zip).

## 1. Check the GPU
Should print a GPU name (e.g. Tesla T4). If not: **Runtime → Change runtime type → T4 GPU → Save**, then run this cell again.

In [ ]:
!nvidia-smi -L || echo 'GPU not found: Runtime -> Change runtime type -> T4 GPU'

## 2. Get the AutoMark code from GitHub
If the repository is private, this fails: then zip your `automark` folder (without `.venv` and `data`), and upload it in the next cell instead.

In [ ]:
REPO = 'https://github.com/demonayush11/Automark.git'
!rm -rf /content/automark && git clone -q $REPO /content/automark && echo 'Code downloaded.' || echo 'Clone failed: upload automark.zip in the next cell.'

## 3. Upload your data
A **Choose files** button appears. Select `pages.zip` **or** `training_data.zip`. (If step 2 failed, also select `automark.zip`.)

In [ ]:
import os, zipfile, shutil
from google.colab import files
os.chdir('/content')
uploaded = files.upload()
if os.path.exists('automark.zip') and not os.path.exists('/content/automark/finetune.py'):
    zipfile.ZipFile('automark.zip').extractall('/content/automark_tmp')
    inner = [r for r, d, f in os.walk('/content/automark_tmp') if 'finetune.py' in f][0]
    shutil.move(inner, '/content/automark')
for name in uploaded:
    if name != 'automark.zip':
        shutil.move(name, '/content/automark/' + name)
os.chdir('/content/automark')
print('Code ready:', os.path.exists('finetune.py'), ' Files here:', [f for f in os.listdir() if f.endswith('.zip')])

*(Optional, for big data)* Instead of uploading, put the zip in Google Drive and run:
```python
from google.colab import drive; drive.mount('/content/drive')
!cp /content/drive/MyDrive/pages.zip /content/automark/
```

## 4. Build the training data from photos + text
Only needed for `pages.zip`. Cuts every page into words, matches each word with the typed text, writes `training_data.zip`. Skip this cell if you uploaded `training_data.zip`.

In [ ]:
!pip -q install pypdfium2
!python make_dataset.py --pages pages.zip --out training_data.zip

## 5. Train
Prints the error before training, then one line per pass over the data. Stops by itself when it stops improving. Usually 5–30 minutes.

**Screenshot the output** — the before/after error on students the model never trained on is your result.

In [ ]:
!python finetune.py --data training_data.zip --weights model/htr_weights.npz --chars model/charList.txt --epochs 40

## 6. Download the new weights
On your computer:
1. Rename the old `automark/model/htr_weights.npz` to `htr_weights_original.npz` (backup).
2. Rename the downloaded file to `htr_weights.npz` and put it in `automark/model/`.
3. Restart the app.

In [ ]:
files.download('htr_weights_finetuned.npz')